# Phase 2: Generative AI - Executive Briefings

In this notebook we use a large language model to turn the model results into a short briefing for an HR executive.
The briefing uses the XGBoost prediction (Phase 1), the K-Means segment (Phase 2) and the main drivers of the score.

Run this notebook from the repo root or from the `Generative_AI/` folder.

In [1]:
# imports and paths
import re
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import textstat
import xgboost as xgb
from IPython.display import Markdown, display
from sklearn.cluster import KMeans
from sklearn.metrics import precision_score, recall_score, roc_auc_score

# find the repo root whether we run from the root or from Generative_AI/
ROOT = Path.cwd().parent if Path.cwd().name == "Generative_AI" else Path.cwd()
GENAI_DIR = ROOT / "Generative_AI"
RESULTS_DIR = GENAI_DIR / "test_results"
RESULTS_DIR.mkdir(exist_ok=True)

# make our module importable
sys.path.insert(0, str(GENAI_DIR))
import api_config_template as cfg
from briefing import DRAFT_TEMPLATE, SYSTEM_MESSAGE, build_briefing, fill_template, is_error

RANDOM_STATE = 42
print("Repo root found:", (ROOT / "Dataset").exists())

Repo root found: True


## 1. Generative AI Model Selection & Setup

**Provider: the Groq API**

- The instructor named LLaMA as a suggested model. Groq is a well known free host for Llama, so we chose Groq.
- Groq gives free access with a free API key. We do not need a paid plan or our own GPU.
- Groq is very fast, so the briefing shows up in the dashboard in about one second.
- The official `groq` Python package has a simple chat API (`client.chat.completions.create`).

**Model: `openai/gpt-oss-120b`**

- Before choosing, we listed the models on our Groq account (cell below). There was **no Llama chat model** available.
  The only Llama models were "prompt guard" models. They are small safety filters and cannot write text.
  We also tried the usual Llama names (for example `llama-3.3-70b-versatile`) and the API said they do not exist.
- So we chose `openai/gpt-oss-120b`. The project description names "LLaMA or GPT" and also allows "open-source models".
  `gpt-oss-120b` is an open-weight GPT model from OpenAI (its weights are public), so it fits both points.
  It is the largest text model on our account and it follows instructions well.
- If a Llama model comes back on Groq, we only need to change `MODEL_NAME` in `api_config_template.py`.

**Settings** (in `api_config_template.py`)

| Setting | Value | Why |
|---|---|---|
| Provider | Groq | free and fast |
| Model | `openai/gpt-oss-120b` | open-weight, the best text model on our account |
| Temperature | 0.2 | low, so the briefing is stable and does not change a lot between runs |
| Reasoning effort | low | this model "thinks" before it answers. Low effort keeps it fast. With the default effort it used all the tokens to think and gave an empty answer |
| Max tokens | 600 | the limit for thinking and answer together. The answer itself is about 100 words |
| Timeout | 30 seconds | the dashboard should not wait forever |

**Limits**

- The free tier has rate limits (requests and tokens per minute and per day). If we send too many requests, the API refuses them for a short time. Our test loop waits a little between calls.
- The model can hallucinate, which means it can write facts or numbers that are not true. We reduce this with a system message that says "use only the numbers given", and we check the outputs in Section 4.
- It needs an internet connection and a valid key. Without them the module returns a clear message and the dashboard keeps working.
- The data is sent to an outside service. We only send the model results (probability, risk, segment, drivers), not the candidate's personal data.

In [2]:
# the settings used (the key itself is never shown)
print("Provider:   ", cfg.PROVIDER)
print("Model:      ", cfg.MODEL_NAME)
print("Temperature:", cfg.TEMPERATURE)
print("Reasoning:  ", cfg.REASONING_EFFORT)
print("Max tokens: ", cfg.MAX_TOKENS)
print("API key set:", cfg.key_is_set())

Provider:    Groq
Model:       openai/gpt-oss-120b
Temperature: 0.2
Reasoning:   low
Max tokens:  600
API key set: True


In [3]:
# list the models available on our Groq account
from groq import Groq

if cfg.key_is_set():
    try:
        client = Groq(api_key=cfg.get_api_key())
        model_ids = sorted(m.id for m in client.models.list().data)
        print("All models on the account:")
        for m in model_ids:
            print(" -", m)
        llama_ids = [m for m in model_ids if "llama" in m.lower()]
        print("Llama models:", llama_ids)
        print("Chosen model is available:", cfg.MODEL_NAME in model_ids)
    except Exception as err:
        print("Could not list the models:", type(err).__name__)
else:
    print("No API key found. Add GROQ_API_KEY to the .env file in the repo root.")

All models on the account:
 - allam-2-7b
 - canopylabs/orpheus-arabic-saudi
 - canopylabs/orpheus-v1-english
 - meta-llama/llama-prompt-guard-2-22m
 - meta-llama/llama-prompt-guard-2-86m
 - openai/gpt-oss-120b
 - openai/gpt-oss-20b
 - openai/gpt-oss-safeguard-20b
 - qwen/qwen3.8-27b
 - whisper-large-v3
 - whisper-large-v3-turbo
Llama models: ['meta-llama/llama-prompt-guard-2-22m', 'meta-llama/llama-prompt-guard-2-86m']
Chosen model is available: True


## 2. Prompt Template Design Documentation

To be completed.

## 3. Implementation & API Integration Code

The code is in `Generative_AI/briefing.py`. The main function is:

```python
build_briefing(ctx, template=None) -> str
```

`ctx` is the same dictionary that the dashboard sends in `Dashboard/genai_panel.py`:
`probability`, `threshold`, `risk`, `cluster`, `cluster_rate` and `drivers` (a list of dicts with `label` and `value`).

What the function does:

1. It fills the template placeholders (`{probability}`, `{threshold}`, `{risk}`, `{cluster}`, `{cluster_rate}`, `{drivers}`) from `ctx`.
2. It sends a system message and the filled prompt to the model on Groq.
3. It returns the text. If anything fails (no key, no internet, rate limit, wrong input), it returns a message that starts with "Briefing not available:" and does not crash.

**Safe key handling**

- The key is read from the environment variable `GROQ_API_KEY`. `python-dotenv` loads it from a `.env` file in the repo root.
- `.env` is listed in `.gitignore`, so the key is never uploaded to GitHub. `Generative_AI/.env.example` shows the format with no real key.
- The key is never printed, logged or saved. The notebook only prints True or False for "key set".
- If an error message ever contains the key, it is replaced with `[hidden]`.

**Fairness rule in the code:** any driver whose label starts with "Gender" is removed before the prompt is built, and other keys in `ctx` (like `candidate`) are never sent to the model.

Below is the system message and the **draft** template.
The draft template is only for testing the framework. It will be replaced by the final templates from Section 2.

In [4]:
# system message used for every call
print(SYSTEM_MESSAGE)
print()
# DRAFT - for testing the framework only, to be replaced by the final templates
print(DRAFT_TEMPLATE)

You write short briefings for an HR executive about one candidate's risk of looking for a job change. Use only the numbers and facts given in the message. Do not invent any facts, numbers, names or reasons. If something is not given, do not guess it. Each driver shows the candidate's real value. Describe the value exactly as given; do not infer it from the direction. Keep it short and clear: at most 120 words, plain business English. Never use gender, or any other personal trait like age, religion or nationality, as a reason or a recommendation. The prediction is a support tool, not a final decision.

Write an executive briefing for this candidate.
Predicted chance of looking for a job change: {probability} (decision threshold {threshold}).
Risk level: {risk}.
Segment: {cluster}, where {cluster_rate} of candidates looked for a job change.
Main drivers of the score:
{drivers}
Give: 1) the assessment in one sentence, 2) the main reasons, 3) one suggested HR action.


In [5]:
# example ctx in the same format as the dashboard
example_ctx = {
    "probability": 0.71, "threshold": 0.32, "risk": "High",
    "cluster": "Segment 4", "cluster_rate": 0.50,
    "drivers": [
        {"label": "City development index: 0.62 (below average, average 0.83)", "value": 0.85},
        {"label": "Gender: Male", "value": 0.05},
        {"label": "Relevant experience: yes", "value": -0.20},
    ],
}

# the filled prompt (gender is removed from the drivers)
print(fill_template(example_ctx))

Write an executive briefing for this candidate.
Predicted chance of looking for a job change: 71.0% (decision threshold 0.32).
Risk level: High.
Segment: Segment 4, where 50% of candidates looked for a job change.
Main drivers of the score:
- City development index: 0.62 (below average, average 0.83) -> raises risk (effect +0.85)
- Relevant experience: yes -> lowers risk (effect -0.20)
Give: 1) the assessment in one sentence, 2) the main reasons, 3) one suggested HR action.


In [6]:
# error handling checks: these must return a message and not crash
print(build_briefing({"risk": "High"}))
print(build_briefing(example_ctx, model="model-that-does-not-exist"))

Briefing not available: the input is missing a value (KeyError: 'probability').
Briefing not available: the API call failed (NotFoundError: Error code: 404 - {'error': {'message': 'The model `model-that-does-not-exist` does not exist or you do not have access to it. Please pass the full model_id including any prefixes.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}).


## 4. Testing Framework & Output Comparison

### 4.1 Test cases from the test set

We use real candidates from `Dataset/preprocessed_test.csv`.
To get their numbers we train the final XGBoost model inside this notebook with the best parameters from Phase 1
(`learning_rate=0.1`, `max_depth=5`, `n_estimators=50`, `subsample=1.0`) and the decision threshold 0.32.
We do not save the model here.

In [7]:
# load the train and test sets
train = pd.read_csv(ROOT / "Dataset" / "preprocessed_train.csv")
test = pd.read_csv(ROOT / "Dataset" / "preprocessed_test.csv")
X_train, y_train = train.drop(columns="target"), train["target"]
X_test, y_test = test.drop(columns="target"), test["target"]

# final XGBoost with the best parameters from Phase 1
THRESHOLD = 0.32
xgb_model = xgb.XGBClassifier(learning_rate=0.1, max_depth=5, n_estimators=50, subsample=1.0,
                              random_state=RANDOM_STATE, eval_metric="logloss")
xgb_model.fit(X_train, y_train)

# quick check that the model behaves as in Phase 1
proba = xgb_model.predict_proba(X_test)[:, 1]
pred = (proba >= THRESHOLD).astype(int)
print(f"ROC-AUC:   {roc_auc_score(y_test, proba):.3f}")
print(f"Recall:    {recall_score(y_test, pred):.3f}")
print(f"Precision: {precision_score(y_test, pred):.3f}")

ROC-AUC:   0.816
Recall:    0.717
Precision: 0.576


In [8]:
# K-Means segments: same settings as the unsupervised notebook (k = 5)
full = pd.read_csv(ROOT / "Dataset" / "preprocessed_data.csv")
clustered = pd.read_csv(ROOT / "Dataset" / "clustered_data.csv")
features = [c for c in full.columns if c != "target"]
kmeans = KMeans(n_clusters=5, n_init=10, random_state=RANDOM_STATE).fit(full[features])

# check that we get the same labels as clustered_data.csv
print("Same labels as clustered_data.csv:", (kmeans.labels_ == clustered["cluster_kmeans"].values).all())

# job change rate in each segment
cluster_rates = clustered.groupby("cluster_kmeans")["target"].mean()
test_clusters = kmeans.predict(X_test[features])
display(cluster_rates.round(3).rename("job_change_rate").to_frame())

Same labels as clustered_data.csv: True


,job_change_rate
cluster_kmeans,
0,0.159
1,0.301
2,0.169
3,0.143
4,0.504


In [9]:
# readable values of the test candidates
# preprocessed_test.csv comes from the same split as Phase 1, and clustered_data.csv
# has the same row order as preprocessed_data.csv, so we can find each test row there
from sklearn.model_selection import train_test_split

_, X_split, _, _ = train_test_split(full[features], full["target"], test_size=0.2,
                                    stratify=full["target"], random_state=RANDOM_STATE)
print("Split matches preprocessed_test.csv:", np.allclose(X_split.values, X_test[features].values))
raw_test = clustered.iloc[X_split.index].reset_index(drop=True)
print("Targets match:", (raw_test["target"].values == y_test.values).all())

CDI_MEAN = clustered["city_development_index"].mean()
ONE_HOT = ["gender", "enrolled_university", "major_discipline", "company_type"]


def driver_label(feature, i):
    # feature name + the candidate's real value, written without double negatives
    # (a label like "No company information: no" made the model drop the "no")
    r = raw_test.iloc[i]
    x = X_test.iloc[i]
    size = "unknown" if r["company_size"] == "Unknown" else f"{r['company_size']} employees"
    edu = "unknown" if r["education_level"] == "Unknown" else r["education_level"]
    if feature == "city_development_index":
        level = "above average" if r["city_development_index"] >= CDI_MEAN else "below average"
        return f"City development index: {r['city_development_index']:.2f} ({level}, average {CDI_MEAN:.2f})"
    if feature == "experience":
        return f"Years of experience: {r['experience']}"
    if feature == "last_new_job":
        if r["last_new_job"] == "never":
            return "Years since last job change: never changed jobs"
        return f"Years since last job change: {r['last_new_job']}"
    if feature == "training_hours_log":
        return f"Training hours: {r['training_hours']}"
    if feature == "company_size":
        return f"Company size: {size}"
    if feature == "education_level":
        return f"Education level: {edu}"
    if feature == "relevent_experience":
        return "Relevant experience: " + ("yes" if x[feature] == 1 else "no")
    if feature == "company_info_missing":
        return "Company information: " + ("missing" if x[feature] == 1 else "provided")
    if feature == "company_size_unknown":
        return "Company size: " + ("unknown" if x[feature] == 1 else f"known ({size})")
    if feature == "education_unknown":
        return "Education level: " + ("unknown" if x[feature] == 1 else f"known ({edu})")
    for prefix in ONE_HOT:
        if feature.startswith(prefix + "_"):
            name = prefix.replace("_", " ").capitalize()
            category = feature[len(prefix) + 1:]
            if x[feature] == 1:
                return f"{name}: {category}"
            return f"{name}: {r[prefix]} (not {category})"
    return f"{pretty(feature)}: {x[feature]}"

Split matches preprocessed_test.csv: True
Targets match: True


In [10]:
# same risk levels and driver labels as the dashboard (Dashboard/utils.py and app.py)
def risk_level(prob, threshold):
    if prob >= max(threshold * 1.5, 0.6):
        return "High"
    return "Elevated" if prob >= threshold else "Low"


def pretty(feature):
    base = {"city_development_index": "City development index", "experience": "Years of experience",
            "last_new_job": "Years since last job change", "training_hours_log": "Training hours",
            "company_size": "Company size", "education_level": "Education level",
            "relevent_experience": "Has relevant experience", "company_size_unknown": "Company size unknown",
            "education_unknown": "Education unknown", "company_info_missing": "No company information"}
    if feature in base:
        return base[feature]
    for prefix in ("gender", "enrolled_university", "major_discipline", "company_type"):
        if feature.startswith(prefix + "_"):
            return f"{prefix.replace('_', ' ').capitalize()}: {feature[len(prefix) + 1:]}"
    return feature


def make_ctx(i, top=8):
    # per-feature contributions of XGBoost for one test row
    x = X_test.iloc[[i]]
    contrib = xgb_model.get_booster().predict(xgb.DMatrix(x), pred_contribs=True)[0][:-1]
    contrib = pd.Series(contrib, index=X_test.columns)
    # drop gender and zero effects, keep the strongest ones
    contrib = contrib[(contrib.abs() > 1e-9) & ~contrib.index.str.startswith("gender")]
    contrib = contrib.reindex(contrib.abs().sort_values(ascending=False).index)[:top]
    cid = int(test_clusters[i])
    return {
        "probability": float(proba[i]),
        "threshold": THRESHOLD,
        "risk": risk_level(proba[i], THRESHOLD),
        "cluster": f"Segment {cid}",
        "cluster_rate": float(cluster_rates[cid]),
        "model": "XGBoost",
        # label = feature name + the candidate's real value, value = effect on the score
        "drivers": [{"label": driver_label(f, i), "value": float(v)}
                    for f, v in contrib.items()],
    }

In [11]:
# pick one typical candidate for each risk level (closest to the median probability of that level)
# plus one borderline candidate just under the threshold
levels = pd.Series([risk_level(p, THRESHOLD) for p in proba])
picked = {}
for level in ["High", "Elevated", "Low"]:
    idx = np.where(levels == level)[0]
    picked[level.lower()] = idx[np.argmin(np.abs(proba[idx] - np.median(proba[idx])))]
below = np.where(proba < THRESHOLD)[0]
picked["borderline"] = below[np.argmax(proba[below])]

cases = [{"case_id": f"{name}_row{i}", "row": int(i), "ctx": make_ctx(i)} for name, i in picked.items()]

# show the cases
pd.DataFrame([{"case_id": c["case_id"], "probability": round(c["ctx"]["probability"], 3),
               "risk": c["ctx"]["risk"], "segment": c["ctx"]["cluster"],
               "segment_rate": round(c["ctx"]["cluster_rate"], 3),
               "top_driver": c["ctx"]["drivers"][0]["label"],
               "actual_target": int(y_test.iloc[c["row"]])} for c in cases])

,case_id,probability,risk,segment,segment_rate,top_driver,actual_target
0,high_row651,0.635,High,Segment 4,0.504,"City development index: 0.62 (below average, a...",1
1,elevated_row242,0.553,Elevated,Segment 3,0.143,Company information: missing,1
2,low_row1668,0.093,Low,Segment 3,0.143,"City development index: 0.88 (above average, a...",0
3,borderline_row817,0.319,Low,Segment 1,0.301,Company information: missing,1


### 4.2 The testing function

`run_tests(templates, cases)` runs every template on every case and measures each answer:

- **response_words**: length of the answer in words. An executive briefing should be short.
- **fk_grade**: Flesch-Kincaid grade level (from `textstat`). Lower means easier to read.
- **keyword_alignment**: share of a small HR keyword list that appears in the answer. It shows if the answer uses the HR domain language.
  We only count keywords that are **not** already in the prompt (system message and filled template), because copying a word from the prompt does not show domain knowledge.
- **response_time_s**: how long the API call took in seconds.
- **call_ok**: False if the call failed (then the metrics are empty).

It waits a little between calls because of the free-tier rate limits.

In [12]:
# small list of HR words we expect in a good briefing
HR_KEYWORDS = ["retention", "risk", "candidate", "job change", "segment", "driver",
               "development", "career", "training", "experience", "engagement",
               "manager", "review", "action", "hr"]


def has_word(word, text):
    # whole word match, not case sensitive
    return re.search(r"\b" + re.escape(word) + r"\b", text.lower()) is not None


def keyword_alignment(text, prompt):
    # only keywords that are NOT in the prompt, so copied words do not count
    new_words = [k for k in HR_KEYWORDS if not has_word(k, prompt)]
    if not new_words:
        return np.nan
    return sum(has_word(k, text) for k in new_words) / len(new_words)


def summarize_input(ctx):
    # one short line that describes the case
    top = ", ".join(d["label"] for d in ctx["drivers"][:2])
    return (f"p={ctx['probability']:.3f}, {ctx['risk']}, {ctx['cluster']} "
            f"({ctx['cluster_rate']:.0%}), top drivers: {top}")


def run_tests(templates, cases, pause=2.0):
    # templates: dict of name -> template text, cases: list of dicts with case_id and ctx
    rows = []
    for t_name, t_text in templates.items():
        for case in cases:
            start = time.perf_counter()
            answer = build_briefing(case["ctx"], template=t_text)
            seconds = time.perf_counter() - start
            ok = not is_error(answer)
            # everything the model saw: system message + filled template
            prompt = SYSTEM_MESSAGE + "\n" + fill_template(case["ctx"], t_text)
            rows.append({
                "template": t_name,
                "case_id": case["case_id"],
                "input_summary": summarize_input(case["ctx"]),
                "response": answer,
                "response_words": len(answer.split()) if ok else np.nan,
                "fk_grade": textstat.flesch_kincaid_grade(answer) if ok else np.nan,
                "keyword_alignment": keyword_alignment(answer, prompt) if ok else np.nan,
                "response_time_s": round(seconds, 2),
                "call_ok": ok,
            })
            time.sleep(pause)
    return pd.DataFrame(rows)

### 4.3 Running the tests

For now only the **draft** template is tested. When the final templates from Section 2 are ready,
they are added to the `templates` dictionary and the same function compares them side by side.

In [13]:
# DRAFT - for testing the framework only, to be replaced by the final templates
templates = {"draft_template": DRAFT_TEMPLATE}

results = run_tests(templates, cases)
results.to_csv(RESULTS_DIR / "test_results.csv", index=False)
print("Saved to Generative_AI/test_results/test_results.csv")
print("Successful calls:", int(results["call_ok"].sum()), "of", len(results))

results.drop(columns="response").round(3)

Saved to Generative_AI/test_results/test_results.csv
Successful calls: 4 of 4


,template,case_id,input_summary,response_words,fk_grade,keyword_alignment,response_time_s,call_ok
0,draft_template,high_row651,"p=0.635, High, Segment 4 (50%), top drivers: C...",95,8.113,0.333,1.06,True
1,draft_template,elevated_row242,"p=0.553, Elevated, Segment 3 (14%), top driver...",58,11.041,0.333,1.68,True
2,draft_template,low_row1668,"p=0.093, Low, Segment 3 (14%), top drivers: Ci...",74,9.463,0.333,2.82,True
3,draft_template,borderline_row817,"p=0.319, Low, Segment 1 (30%), top drivers: Co...",84,8.457,0.000,1.64,True


In [14]:
# comparison table: average metrics per template
results.groupby("template")[["response_words", "fk_grade", "keyword_alignment", "response_time_s"]].mean().round(2)

,response_words,fk_grade,keyword_alignment,response_time_s
template,,,,
draft_template,77.75,9.27,0.25,1.8


In [15]:
# one example output
example = results.iloc[0]
print("Template:", example["template"])
print("Case:    ", example["case_id"])
print("Input:   ", example["input_summary"])
display(Markdown(example["response"]))

Template: draft_template
Case:     high_row651
Input:    p=0.635, High, Segment 4 (50%), top drivers: City development index: 0.62 (below average, average 0.83), Company information: missing


**Assessment:** The candidate has a 63.5% predicted chance of looking for a job change, placing them in the High risk level (Segment 4, where 50% of candidates looked for a change).  

**Main reasons:** City development index 0.62 (below the average 0.83) adds +0.91 risk; missing company information adds +0.32; less than one year of experience adds +0.23; unknown company size adds +0.11; the only mitigating factor is never having changed jobs (-0.13).  

**Suggested HR action:** Initiate a proactive engagement conversation to understand the candidate’s career goals and address any concerns about location or company fit.

## 5. Analysis: Qualitative & Quantitative Results

## 6. Best Prompt Selection & Justification

## 7. Integration Plan for Final System

## 8. Ethical Considerations & Limitations